In [ ]:
pip install -e /root/capsule

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import glob
import pickle
import os
import logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(filename)s:%(lineno)d - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
SESSION_KEYS = ["subject_id", "session_date"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
import seaborn as sns

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model import ForagerCollection

from aind_analysis_arch_result_access.han_pipeline import get_session_table, get_mle_model_fitting
from aind_analysis_arch_result_access.util.s3 import get_s3_pkl, get_s3_json

import aind_dynamic_foraging_population_analysis

In [ ]:
from pynwb import NWBHDF5IO

LOCAL_NWB_TMP = "/data/foraging_nwb_bonsai"

def get_nwb_from_local_tmp(session_id):
    """Get NWB file from session_id.

    Overwrite this function to get NWB file from other places.

    Parameters
    ----------
    session_id : _type_
        _description_
    """
    io = NWBHDF5IO(f"{LOCAL_NWB_TMP}/{session_id}.nwb", mode="r")
    nwb = io.read()
    return nwb


def get_history_from_nwb(nwb):
    """Get choice and reward history from nwb file
    
    #TODO move this to aind-behavior-nwb-util
    """

    df_trial = nwb.trials.to_dataframe()

    p_reward = [
        df_trial.reward_probabilityL.values,
        df_trial.reward_probabilityR.values,
    ]
    baiting = False if "without baiting" in nwb.protocol.lower() else True

    choice_history = df_trial.animal_response.map({0: 0, 1: 1, 2: np.nan}).values
    reward_history = df_trial.rewarded_historyL | df_trial.rewarded_historyR
    
    # autowater_offered = (df_trial.auto_waterL == 1) | (df_trial.auto_waterR == 1)
    # random_number = [
    #     df_trial.reward_random_number_left.values,
    #     df_trial.reward_random_number_right.values,
    # ]

    

    return (
        baiting,
        p_reward,
        choice_history,
        reward_history
        # autowater_offered,
        # random_number,
    )

In [ ]:
def get_all_model_metrics(
    use_cache=True, cache_path="~/capsule/data/df_model_fitting_all.pkl"
):
    """Get all model metrics from either cache or result access API.

    Parameters
    ----------
    use_cache : bool, optional
        Whether to use cached data, by default True
        If true, it will load data from the cache. If cache does not exist
           or is invalid, it will fetch data from the API.
        If False, it will fetch data from the API and update the cache_path.
    cache_path : str, optional
        Cache path, by default "~/capsule/data/df_model_fitting_all.pkl"
    """

    if use_cache:
        try:
            logger.info(f"Trying to load data from cache: {cache_path}...")
            df_model_fitting = pd.read_pickle(cache_path)
            logger.info(f"{len(df_model_fitting)} rows loaded from cache.")
            return df_model_fitting
        except Exception as e:
            logger.warning(f"Cache not found or invalid: {e}. Fetching from API.")

    # Fetch from result access API
    logger.info("Fetching data from result access API...")
    df_model_fitting = get_mle_model_fitting(
        from_custom_query={"status": "success"},
        if_include_latent_variables=False,
        paginate_settings={"paginate": True, "paginate_batch_size": 5000},
    )
    df_model_fitting.to_pickle(cache_path)
    logger.info(f"{len(df_model_fitting)} rows fetched from API and saved to cache.")
    return df_model_fitting


def enrich_with_df_session(df, selected_fields):
    """Enrich any df with session information from get_session_table.

    Parameters
    ----------
    df: pd.DataFrame
        Any dataFrame containing SESSION_KEYS (["subject_id", "session_date"])
    selected_fields: list of str
        Fields to merge from session table.

    Returns
    -------
    pd.DataFrame
        Enriched DataFrame with selected session information.
    """
    logger.info("Fetching session table...")
    df_session = get_session_table()

    logger.info("Merging model fitting data with session data...")
    # Merge in session metadata
    df_session["session_date"] = df_session["session_date"].astype("str")
    df_enriched = df.merge(
        df_session[SESSION_KEYS + selected_fields],
        on=SESSION_KEYS,
        how="left",
    )
    return df_enriched

In [ ]:
# # load fitted session data
# # takes ~8min to run

# df_model_fitting = get_all_model_metrics(use_cache=True, 
#                                          cache_path=os.path.expanduser("~/capsule/results/df_model_fitting_all.pkl"))
# df_model_fitting = enrich_with_df_session(
#     df_model_fitting,
#     selected_fields=[
#         "nwb_suffix",
#         "curriculum_name",
#         "curriculum_version_group",
#         "current_stage_actual",
#     ],
# )

In [ ]:
# # save the df to for fast access
# df_model_fitting.to_pickle(
#     os.path.expanduser("~/capsule/analysis_result/df_model_fitting_all_250714.pkl")
# )

In [ ]:
# load previously saved fitted session data
df_model_fitting = pd.read_pickle(
    os.path.expanduser("~/capsule/analysis_result/df_model_fitting_all_250714.pkl")
)

print(f"Loaded {len(df_model_fitting)} rows of model fitting data.")

In [ ]:
df_model_fitting.columns
# df_model_fitting.head()

In [ ]:
# # run and saved on 250714

# # get relevant fitted data information
# # session_id, curriculum_type, stage, fitted_params
# # save the df to ~/capsule/results/df_model_fitting_relevant_{date}.pkl for fast access

# # retain only relevant models
# models_to_keep = [
#     'ForagingCompareThreshold',
#     'QLearning_L2F1_softmax',  # Hattori
#     'QLearning_L1F1_CK1_softmax'  # Bari
# ]
# df_model_fitting_relevant = df_model_fitting[df_model_fitting['agent_alias'].isin(models_to_keep)]

# # retain only relevant columns
# columns_to_keep = [
#     '_id', 
#     "subject_id",
#     "session_date",
#     "nwb_suffix",
#     'nwb_name',
#     "curriculum_name",
#     "curriculum_version_group",
#     "current_stage_actual",
#     "n_trials",
#     "agent_alias",
#     "params",
#     "log_likelihood",
#     "AIC",
#     "BIC",
#     "prediction_accuracy_test",
#     "prediction_accuracy_10-CV_test",
# ]
# df_model_fitting_relevant = df_model_fitting_relevant[columns_to_keep]

# # sort by subject_id and session_date
# df_model_fitting_relevant = df_model_fitting_relevant.sort_values(
#     by=["subject_id", "session_date", "agent_alias"]
# )


# # save the df to for fast access
# df_model_fitting_relevant.to_pickle(
#     os.path.expanduser("~/capsule/analysis_result/df_model_fitting_relevant_250714.pkl")
# )

In [ ]:
# if already saved, load the df
df_model_fitting_relevant = pd.read_pickle(
    os.path.expanduser("~/capsule/analysis_result/df_model_fitting_relevant_250714.pkl")
)

print(f"Loaded {len(df_model_fitting_relevant)} rows of relevant model fitting data.")

In [ ]:
df_model_fitting_relevant.head()

## get behavioral data for fitted sessions

In [ ]:
# # run and saved on 250714
# # takes quite long, ~2.5 hour
# import time

# # per session, get choice and reward history from nwb file

# # get unique session nwb_names
# def get_unique_nwb_names(df_model_fitting):
#     """Get unique nwb_names from df_model_fitting."""
#     return df_model_fitting['nwb_name'].unique().tolist()

# # for each nwb_name in the list, get the session history
# def get_session_history_from_nwb(df_model_fitting):
#     """Get session history from NWB files for each session in df_model_fitting."""
#     session_history = []
#     unique_nwb_names = get_unique_nwb_names(df_model_fitting)
    
#     print(f"Found {len(unique_nwb_names)} unique NWB files to process.")

#     for nwb_name in unique_nwb_names:
        
#         # record time taken to process each NWB file
#         start_time = time.time()

#         # try to get nwb file from local tmp directory
#         try:
#             nwb = get_nwb_from_local_tmp(nwb_name.split('.')[0])
#         except Exception as e:
#             print(f"ERROR LOADING NWB {nwb_name}: {e}")
#             continue

#         baiting, p_reward, choice_history, reward_history = get_history_from_nwb(nwb)
#         end_time = time.time()
#         print(f"{nwb_name}: processed in {end_time - start_time:.2f}s.")

#         session_history.append({
#             "nwb_name": nwb_name,
#             "baiting": baiting,
#             "p_reward": p_reward,
#             "choice_history": choice_history,
#             "reward_history": reward_history
#         })
#     return pd.DataFrame(session_history)

# # get session history from nwb files
# df_session_history = get_session_history_from_nwb(df_model_fitting_relevant)

In [ ]:
# # run and saved on 250714

# # expand the session history to include 
# # subject_id, session_date, curriculum_name, curriculum_version_group, current_stage_actual
# def expand_session_history(df_model_fitting, df_session_history):
#     """
#     Expand session history to include 
#     subject_id, session_date, 
#     curriculum_name, curriculum_version_group, current_stage_actual,
#     n_trials
#     from df_model_fitting.
#     """
#     # Deduplicate df_model_fitting by nwb_name, keeping the first occurrence
#     df_model_fitting_unique = df_model_fitting.drop_duplicates(subset=["nwb_name"])
    
#     df_session_history_expanded = df_session_history.merge(
#         df_model_fitting_unique[SESSION_KEYS + ["nwb_name", 
#             "curriculum_name", "curriculum_version_group", "current_stage_actual",
#             "n_trials"
#         ]],
#         left_on="nwb_name",
#         right_on="nwb_name",
#         how="left"
#     )
#     return df_session_history_expanded

# # expand the session history
# df_session_history_expanded = expand_session_history(
#     df_model_fitting_relevant, df_session_history
# )

In [ ]:
# # save the session history for fast access
# df_session_history_expanded.to_pickle(
#     os.path.expanduser("~/capsule/analysis_result/df_session_history_250714.pkl")
# )

In [ ]:
# if already saved, load the df
df_session_history = pd.read_pickle(
    os.path.expanduser("~/capsule/analysis_result/df_session_history_250714.pkl")
)

print(f"Loaded {len(df_session_history)} rows of session history data.")

In [ ]:
df_session_history.head()

# analyze run length distribution

In [ ]:
from typing import Tuple

def extract_switches_and_run_lengths(choices: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extract switch trial indices and their associated run lengths.

    Parameters:
    -----------
    choices : np.ndarray
        Array of binary choices (0 or 1) for each trial
        
    Returns:
    --------
    switch_indices : np.ndarray
        Indices where switches occur (0-based)
    run_lengths : np.ndarray
        Run length preceding each switch
    """
    # Ensure choices is a numpy array
    choices = np.asarray(choices)

    # drop NaN values if present
    choices = choices[~np.isnan(choices)]
    
    if len(choices) < 2:
        raise ValueError("Choices array must have at least two elements to find switches.")
        return np.array([]), np.array([])
    
    
    # Find where consecutive choices differ
    switches = np.diff(choices) != 0
    switch_indices = np.where(switches)[0] + 1  # +1 because diff reduces length by 1
    
    # Calculate run lengths efficiently using vectorized operations
    if len(switch_indices) > 0:
        # Run lengths are differences between consecutive switch indices
        # First run length is from start (0) to first switch
        run_lengths = np.diff(np.concatenate(([0], switch_indices)))
    else:
        run_lengths = np.array([])
    
    # check if switch_indices and run_lengths are of the same length
    if len(switch_indices) != len(run_lengths):
        raise ValueError("Switch indices and run lengths should have the same length. " + 
                         f"Instead got {len(switch_indices)} and {len(run_lengths)}.")

    return switch_indices, run_lengths

In [ ]:
# per session, get switch trial indices and run lengths
# insert switch trial indices and run lengths into df_session_history
def insert_switch_info(df_session_history):
    """Insert switch trial indices and run lengths into df_session_history."""
    switch_indices_list = []
    run_lengths_list = []
    
    for _, row in df_session_history.iterrows():
        choices = row['choice_history']
        if choices is not None:
            switch_indices, run_lengths = extract_switches_and_run_lengths(choices)
            switch_indices_list.append(switch_indices)
            run_lengths_list.append(run_lengths)
        else:
            switch_indices_list.append(np.array([]))
            run_lengths_list.append(np.array([]))
    
    df_session_history['switch_indices'] = switch_indices_list
    df_session_history['run_lengths'] = run_lengths_list
    
    return df_session_history

In [ ]:
df_session_history = insert_switch_info(df_session_history)

In [ ]:
df_session_history.columns

In [ ]:
# definte a function to plot run lengths distribution for a given df

def plot_run_lengths_distribution(df):
    """
    Plot the distribution of run lengths from a DataFrame
    with 2 subplots: one for the histogram and one for the violin plot.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing 'run_lengths' column.
    """
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=300)

    # Histogram
    axes[0].hist(
        np.concatenate(df['run_lengths'].dropna().values),
        bins=np.arange(0, 400, 1),  # Adjust bin size as needed
        density=False,
        alpha=0.6,
        color='blue',
    )
    axes[0].set_title("Run length distribution (Histogram)")
    axes[0].set_xlabel("Run length (inter-switch interval)")
    axes[0].set_ylabel("Count")
    axes[0].set_yscale('log')

    # Violin plot
    mean_run_length = np.mean(np.concatenate(df['run_lengths'].dropna().values))
    std_run_length = np.std(np.concatenate(df['run_lengths'].dropna().values))
    median_run_length = np.median(np.concatenate(df['run_lengths'].dropna().values))
    axes[1].axvline(mean_run_length, color='red', linestyle='-', label=f'Mean+/-Std: {mean_run_length:.2f}+/-{std_run_length:.2f}')
    axes[1].axvline(median_run_length - std_run_length, color='orange', linestyle='--')
    axes[1].axvline(median_run_length + std_run_length, color='orange', linestyle='--')
    axes[1].axvline(median_run_length, color='green', linestyle='-', label=f'Median: {median_run_length:.2f}')

    sns.violinplot(
        x=np.concatenate(df['run_lengths'].dropna().values),
        ax=axes[1],
        color='lightblue'
    )
    # put mean and median lines
    
    axes[1].set_title("Run length distribution (Violin plot)")
    axes[1].set_xlabel("Run length (inter-switch interval)")
    axes[1].set_ylabel("Density")

    plt.tight_layout()
    plt.show()


In [ ]:

task_types = [
    'Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting', 'None'
]

later_stages = [
    'STAGE_FINAL', 'GRADUATED'
]

# Plot run lengths distribution for all sessions
plot_run_lengths_distribution(df_session_history)

# Plot run lengths distribution for each task type
for task_type in task_types:
    print(f"Plotting run lengths distribution for {task_type}...")
    plot_run_lengths_distribution(df_session_history[
        (df_session_history['curriculum_name'] == task_type) &
        (df_session_history['current_stage_actual'].isin(later_stages))]
    )

In [ ]:
df_session_history.head()

## compare with simulated data

In [ ]:
# load saved simulation results
saved_file = os.path.expanduser("~/capsule/analysis_result/dict_simulation_results_250716.pkl")
with open(saved_file, 'rb') as f:
    dict_simulation_results = pickle.load(f)

In [ ]:

model_types = [
    'ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax'
]

task_types = [
    'Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting', 'None'
]

for model_type in model_types:
    for task_type in task_types:
        key = (model_type, task_type)
        print(f"Simulation results for {key}...")
        df_simulation_results = dict_simulation_results[key]

        # get switch trial indices and run lengths
        df_simulation_results = insert_switch_info(df_simulation_results)

        # plot run lengths distribution
        plot_run_lengths_distribution(df_simulation_results)

# p(switch) pre- and post- switch

In [ ]:
# define a function to plot p(switch) pre- and post-switch given a df
# condition on a switch trial index
# get the choice sequence for 5 trials before and after the switch trial index
# aggregate over switch trials, sessions
# for each trial, calculate the probability of switching
def plot_switch_probabilities(df, switch_indices_col='switch_indices', 
                              choice_history_col='choice_history', 
                              n_trials=5):
    """
    Plot the probability of switching pre- and post-switch trials.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing switch indices and choice history.
    switch_indices_col : str, optional
        Column name for switch indices, by default 'switch_indices'
    choice_history_col : str, optional
        Column name for choice history, by default 'choice_history'
    n_trials : int, optional
        Number of trials to consider before and after the switch, by default 5
    """
    
    # Initialize a list to collect probabilities
    probabilities = []

    # Iterate through each row in the DataFrame
    for _, row in df.iterrows():
        switch_indices = row[switch_indices_col]
        choices = row[choice_history_col]

        if len(switch_indices) == 0 or len(choices) < n_trials * 2:
            continue  # Skip if no switches or not enough trials

        for switch_index in switch_indices:
            # Get the range of trials around the switch index
            start_index = max(0, switch_index - n_trials)
            end_index = min(len(choices), switch_index + n_trials + 1)

            # Get the choices in this range
            trial_choices = choices[start_index:end_index]

            # Calculate the probability of switching
            if len(trial_choices) < n_trials * 2:
                continue  # Skip if not enough trials

            pre_switch_choices = trial_choices[:n_trials]
            post_switch_choices = trial_choices[n_trials:]

            p_switch_pre = np.mean(pre_switch_choices)
            p_switch_post = np.mean(post_switch_choices)

            probabilities.append({
                'p_switch_pre': p_switch_pre,
                'p_switch_post': p_switch_post,
                'session_id': row['nwb_name'],
                'task_type': row['curriculum_name'],
                'model_type': row['agent_alias']
            })

    # Convert to DataFrame
    df_probabilities = pd.DataFrame(probabilities)

    # Plotting
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_probabilities, x='task_type', y='p_switch_pre', hue='model_type')
    plt.title('Probability of Switching Pre-Switch Trials')

